# Sesión 2 · Valor del dinero en el tiempo y CETES

**Módulo 1: Aplicaciones de Python en finanzas** · Tema 2.1 · sábado 21 de noviembre de 2026

Al terminar esta sesión podrás:

1. Calcular valor futuro y valor presente con interés simple, compuesto y continuo.
2. Convertir tasas entre sí (nominal, efectiva y continua).
3. Calcular el precio y el rendimiento de un CETE, y comprobarlo contra el ejemplo oficial de Banco de México.
4. Valuar una serie de flujos (VPN y TIR) y usarlo para comparar alternativas de inversión o de crédito.
5. Dejar tus funciones documentadas y probadas en `src/finanzas.py` (Práctica 2).

> **Antes de empezar.** Ve al menú **Archivo → Guardar una copia en Drive**. Trabaja siempre en tu copia; si no, tus cambios se pierden al cerrar la pestaña.

**Cómo leer este notebook**

- Las celdas con `# Tu código aquí` son ejercicios: ahí escribes tú.
- Después de cada ejercicio hay una celda **Comprueba tu respuesta**. Si no marca error, vas bien.
- Si una celda de comprobación falla, es normal que las siguientes también fallen: regresa al ejercicio y corrígelo.
- La notación es la del [formulario del módulo](https://github.com/nana-bug/m1-python-finanzas-2026-2/blob/main/docs/formulario.md): tasas en decimales (0.05, no 5), `n` en años, `m` capitalizaciones por año.

In [ ]:
# CELDA ESTÁNDAR · Córrela siempre primero.
# Prepara la carpeta de trabajo: en Colab descarga el repositorio de la clase;
# en tu computadora se coloca en la raíz del repositorio.
import os
import subprocess
import sys

REPO = "m1-python-finanzas-2026-2"
URL = f"https://github.com/nana-bug/{REPO}.git"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", URL, f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"], check=False)
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy-financial"], check=True)
else:
    while not os.path.isdir("src") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import math
from typing import Sequence

import matplotlib.pyplot as plt
import numpy_financial as npf

---
## Caso motivador: las tasas de CETES de esta semana

Cada martes, Banco de México subasta CETES (Certificados de la Tesorería de la Federación) por cuenta del Gobierno Federal. Un CETE es el instrumento más simple que existe: pagas hoy un precio menor a 10 pesos y al vencimiento recibes 10 pesos. No paga intereses en el camino.

Estos fueron los resultados de la subasta del 10 de noviembre de 2026:

| Plazo | Tasa de rendimiento anual |
|---|---|
| 28 días | 6.20% |
| 91 días | 6.47% |
| 182 días | 6.61% |
| 364 días | 6.95% |

**La pregunta de hoy.** Tienes 100,000 pesos que no vas a necesitar durante un año. ¿Qué te conviene más?

- **A.** Comprar CETES a 364 días al 6.95%.
- **B.** Comprar CETES a 28 días al 6.20% y renovarlos cada 28 días durante todo el año.

Anota tu respuesta antes de seguir. Para contestarla bien necesitamos tres cosas que veremos en orden: cómo crece el dinero (bloque 1), cómo se calcula el precio de un CETE (bloque 2) y cómo se comparan tasas de plazos distintos (bloque 2).

In [ ]:
# PARÁMETROS DE LA SESIÓN · tasas de rendimiento de la última subasta, en decimales.
# Si ya hay una subasta más reciente, cambia aquí los números y la fecha.
FECHA_SUBASTA = "10/11/2026"
TASAS_CETES = {28: 0.0620, 91: 0.0647, 182: 0.0661, 364: 0.0695}   # plazo en días: tasa anual

---
## Bloque 1 · Cómo crece el dinero

### 1.1 Interés simple

Con interés simple, los intereses se calculan siempre sobre el capital inicial:

$$VF = VP\,(1 + r\,n)$$

$VP$ es el valor presente (lo que tienes hoy), $VF$ el valor futuro, $r$ la tasa anual y $n$ el plazo en años.

In [ ]:
def valor_futuro_simple(vp: float, r: float, n: float) -> float:
    """Valor futuro con interés simple: VF = VP (1 + r n)."""
    return vp * (1 + r * n)


valor_futuro_simple(1_000, 0.10, 2)   # 1,000 pesos al 10% durante 2 años

### 1.2 Interés compuesto

Con interés compuesto, los intereses se suman al capital y también generan intereses. Si la tasa anual $r$ se capitaliza $m$ veces al año:

$$VF = VP\left(1 + \frac{r}{m}\right)^{m\,n}$$

Si capitalizamos cada vez más seguido ($m \to \infty$) llegamos a la **capitalización continua**:

$$VF = VP\,e^{r\,n}$$

En la función, `m=None` significa capitalización continua.

In [ ]:
def valor_futuro(vp: float, r: float, n: float, m: int | None = 1) -> float:
    """Valor futuro con interés compuesto: VF = VP (1 + r/m)^(m n).

    m: capitalizaciones por año; m=None es capitalización continua, VF = VP e^(r n).
    """
    if m is None:
        return vp * math.exp(r * n)
    return vp * (1 + r / m) ** (m * n)

In [ ]:
# El mismo depósito con distintas frecuencias de capitalización
for nombre, m in [("anual", 1), ("semestral", 2), ("mensual", 12), ("diaria", 365), ("continua", None)]:
    print(f"{nombre:>10}: {valor_futuro(100_000, 0.10, 10, m):>12,.2f}")

In [ ]:
# ¿Qué tan rápido se acerca la capitalización compuesta a la continua?
frecuencias = [1, 2, 4, 12, 52, 365]
valores = [valor_futuro(1, 0.10, 10, m) for m in frecuencias]

plt.figure(figsize=(7, 3.5))
plt.plot(range(len(frecuencias)), valores, marker="o", label="Compuesta m veces al año")
plt.axhline(valor_futuro(1, 0.10, 10, None), color="gray", linestyle="--", label="Continua")
plt.xticks(range(len(frecuencias)), frecuencias)
plt.xlabel("Capitalizaciones por año (m)")
plt.ylabel("Valor futuro de 1 peso")
plt.title("1 peso al 10% durante 10 años")
plt.legend()
plt.show()

### 1.3 Valor presente

El valor presente contesta la pregunta inversa: ¿cuánto vale **hoy** una cantidad que recibiré en el futuro?

$$VP = \frac{VF}{\left(1 + \dfrac{r}{m}\right)^{m\,n}}$$

**Ejercicio 1.** Completa la función `valor_presente`. Pista: ya tienes `valor_futuro`; el valor futuro de 1 peso es justo el número entre el que hay que dividir.

In [ ]:
def valor_presente(vf: float, r: float, n: float, m: int | None = 1) -> float:
    """Valor presente de un solo pago: VP = VF / (1 + r/m)^(m n)."""
    # Tu código aquí

In [ ]:
# Comprueba tu respuesta
assert math.isclose(valor_presente(1_210, 0.10, 2), 1_000)
assert math.isclose(valor_presente(100, 0.061, 100, m=2), 0.2457, abs_tol=1e-4)
print("Correcto: valor_presente funciona.")

### 1.4 Tasa efectiva

Dos tasas con distinta frecuencia de capitalización no se pueden comparar directamente. La **tasa efectiva anual** dice cuánto crece 1 peso en un año, ya con la capitalización incluida:

$$r_{ef} = \left(1 + \frac{r}{m}\right)^{m} - 1$$

**Ejercicio 2.** Completa `tasa_efectiva`. Pista: es el valor futuro de 1 peso a un año, menos el peso inicial.

In [ ]:
def tasa_efectiva(r: float, m: int | None) -> float:
    """Tasa efectiva anual: r_ef = (1 + r/m)^m - 1; con m=None, r_ef = e^r - 1."""
    # Tu código aquí

In [ ]:
# Comprueba tu respuesta
assert math.isclose(tasa_efectiva(0.12, 12), 0.1268, abs_tol=1e-4)
assert math.isclose(tasa_efectiva(0.10, None), math.e**0.10 - 1)
print("Correcto: 12% capitalizable cada mes equivale a", f"{tasa_efectiva(0.12, 12):.2%}", "efectivo anual.")

La tasa continua equivalente a una efectiva es $r_c = \ln(1 + r_{ef})$. La usaremos la próxima sesión: es la razón por la que existen los rendimientos logarítmicos.

In [ ]:
def tasa_continua(r_ef: float) -> float:
    """Tasa continua equivalente a una efectiva anual: r_c = ln(1 + r_ef)."""
    return math.log(1 + r_ef)


tasa_continua(0.10)

### Punto de control

**¿En cuántos años se duplica una inversión al 10% anual compuesto?** Despeja $n$ de $2 = (1 + 0.10)^n$ y calcúlalo. Pega tu resultado en el chat.

In [ ]:
# Tu código aquí
print(f"Años para duplicar: {anios_para_duplicar:.2f}")

In [ ]:
# Comprueba tu respuesta
assert math.isclose(valor_futuro(1, 0.10, anios_para_duplicar), 2)
print("Correcto.")

---
## Bloque 2 · CETES

### 2.1 Del rendimiento al precio

Un CETE tiene valor nominal $VN = 10$ pesos. Si le faltan $d$ días para vencer y su tasa de rendimiento anual es $r$, su precio es:

$$P = \frac{VN}{1 + r\,\dfrac{d}{360}}$$

Dos detalles que causan la mayoría de los errores:

- La tasa se aplica con **interés simple** sobre el plazo del CETE.
- El año de los CETES tiene **360 días**, no 365.

Banco de México publica un ejemplo oficial: un CETE a **91 días** con tasa de **4.39%** tiene un precio de **9.8902485** pesos. Si nuestra función da ese número, está bien.

In [ ]:
def precio_cete(r: float, d: int, vn: float = 10.0) -> float:
    """Precio de un CETE a partir de su tasa de rendimiento: P = VN / (1 + r d/360)."""
    return vn / (1 + r * d / 360)


precio_cete(0.0439, 91)

In [ ]:
# Precio de los CETES de la última subasta
for dias, tasa in TASAS_CETES.items():
    print(f"CETES a {dias:>3} días, tasa {tasa:.2%}: precio {precio_cete(tasa, dias):.7f}")

### 2.2 Ejercicio en parejas (20 minutos)

Trabajen de dos en dos, con una sola pantalla compartida. Una persona escribe y la otra revisa; a la mitad cambian.

**Parte A · Del precio al rendimiento.** Despejen $r$ de la fórmula del precio y completen `rendimiento_cete`:

$$r = \left(\frac{VN}{P} - 1\right)\frac{360}{d}$$

In [ ]:
def rendimiento_cete(p: float, d: int, vn: float = 10.0) -> float:
    """Tasa de rendimiento implícita en el precio: r = (VN/P - 1) 360/d."""
    # Tu código aquí

In [ ]:
# Comprueba tu respuesta: ida y vuelta con el ejemplo de Banco de México
assert math.isclose(rendimiento_cete(9.8902485, 91), 0.0439, abs_tol=1e-7)
assert math.isclose(rendimiento_cete(precio_cete(0.0647, 91), 91), 0.0647)
print("Correcto: del precio regresamos a la tasa.")

**Parte B · Verifiquen una cifra de un LLM.** Pregunten a un LLM (ChatGPT, Gemini, Claude, el que usen):

> ¿Cuál es el precio de un CETE a 28 días con una tasa de rendimiento de 6.20%?

Escriban abajo el precio que les dio y compárenlo con el de su función. Si no coincide, averigüen por qué: los errores más comunes son usar 365 días en lugar de 360, o usar interés compuesto.

In [ ]:
PRECIO_LLM = None   # escribe aquí el precio que te dio el LLM, por ejemplo 9.95

# Tu código aquí
print(f"Precio con nuestra función: {precio_verificado:.7f}")
if PRECIO_LLM is not None:
    print(f"Precio del LLM:             {PRECIO_LLM:.7f}")
    print(f"Diferencia:                 {PRECIO_LLM - precio_verificado:+.7f} pesos por título")

In [ ]:
# Comprueba tu respuesta
assert round(precio_verificado, 7) == 9.9520092
print("Correcto.")

**Parte C · La tasa efectiva de renovar un CETE.** Si renuevas un CETE de $d$ días durante un año, reinvirtiendo cada vez, la tasa efectiva anual es:

$$r_{ef} = \left(1 + r\,\frac{d}{360}\right)^{360/d} - 1$$

Completen `tasa_efectiva_cete`.

In [ ]:
def tasa_efectiva_cete(r: float, d: int, base: int = 360) -> float:
    """Tasa efectiva anual de renovar un CETE de d días: (1 + r d/360)^(base/d) - 1."""
    # Tu código aquí

In [ ]:
# Comprueba tu respuesta
assert math.isclose(tasa_efectiva_cete(0.0620, 28), 0.0638, abs_tol=1e-4)
print("Correcto: CETES a 28 días al 6.20% equivalen a", f"{tasa_efectiva_cete(0.0620, 28):.2%}", "efectivo anual.")

### 2.3 Respuesta al caso motivador

Ahora sí podemos comparar las cuatro opciones de la subasta en la misma unidad: tasa efectiva anual.

In [ ]:
for dias, tasa in TASAS_CETES.items():
    print(f"CETES a {dias:>3} días: tasa {tasa:.2%}  ->  efectiva anual {tasa_efectiva_cete(tasa, dias):.2%}")

**¿A o B?**

*Escribe aquí tu respuesta.*

### 2.4 La tasa de descuento (para leer los resultados de la subasta)

Banco de México también publica la **tasa de descuento** $d_s$, que se aplica sobre el valor nominal en lugar de sobre el precio:

$$P = VN\left(1 - d_s\,\frac{d}{360}\right) \qquad\Longleftrightarrow\qquad d_s = \frac{r}{1 + r\,\dfrac{d}{360}}$$

En el ejemplo oficial, la tasa de rendimiento de 4.39% corresponde a un descuento de 4.34%.

In [ ]:
def descuento_cete(r: float, d: int) -> float:
    """Tasa de descuento equivalente a la de rendimiento: d_s = r / (1 + r d/360)."""
    return r / (1 + r * d / 360)


print(f"Descuento del ejemplo de Banxico: {descuento_cete(0.0439, 91):.2%}")

---
## Bloque 3 · Valor presente de varios flujos, VPN y TIR

### 3.1 Un caso extremo: el bono mexicano a 100 años

En octubre de 2010 México colocó un bono en dólares que vence en **2110**. Antes de calcular nada: ¿cuánto crees que valían, el día de la emisión, 100 dólares que el bono pagará dentro de 100 años?

El mercado descontó ese bono a una tasa de 6.1% anual capitalizable cada semestre.

In [ ]:
vp_cien = valor_presente(100, 0.061, 100, m=2)
print(f"100 dólares dentro de 100 años valían {vp_cien:.4f} dólares: unos {vp_cien * 100:.0f} centavos.")

### 3.2 Valor presente de una serie de flujos

Casi ninguna inversión es un solo pago. Si los flujos $C_k$ se pagan en los tiempos $t_k$:

$$VP = \sum_{k} \frac{C_k}{(1 + r)^{t_k}}$$

Cuando el primer flujo es el desembolso inicial (negativo) y ocurre hoy, al resultado se le llama **valor presente neto (VPN)**. Regla de decisión: si el VPN es positivo, el proyecto vale más de lo que cuesta.

Cuidado con las unidades: `r` y `tiempos` deben estar en la misma unidad. Si la tasa es mensual, los tiempos van en meses.

In [ ]:
def valor_presente_flujos(
    flujos: Sequence[float], r: float, tiempos: Sequence[float] | None = None
) -> float:
    """Valor presente de flujos C_k pagados en t_k: VP = sum C_k / (1 + r)^t_k.

    tiempos: en las mismas unidades que r (r mensual -> meses). Por omisión 0, 1, 2, ...
    de modo que el primer flujo ocurre hoy y no se descuenta.
    """
    if tiempos is None:
        tiempos = range(len(flujos))
    return sum(c / (1 + r) ** t for c, t in zip(flujos, tiempos))

In [ ]:
# Un proyecto: invertir 100,000 hoy y recibir cuatro pagos anuales
flujos = [-100_000, 25_000, 30_000, 35_000, 40_000]

vpn = valor_presente_flujos(flujos, 0.10)
print(f"VPN al 10%: {vpn:,.2f}")
print(f"Con numpy-financial: {npf.npv(0.10, flujos):,.2f}")

> **Ojo si vienes de Excel.** `npf.npv` y nuestra función suponen que el primer flujo ocurre hoy y no lo descuentan. La función `VNA` de Excel sí descuenta el primer flujo. Con los mismos números dan resultados distintos.

### 3.3 Tasa interna de retorno (TIR)

La TIR es la tasa $r$ que hace que el valor presente de los flujos sea cero. No se puede despejar; hay que buscarla. El método más simple es la **bisección**: si el VP es positivo en una tasa y negativo en otra, la TIR está en medio; partimos el intervalo a la mitad y repetimos.

In [ ]:
def tir(
    flujos: Sequence[float],
    tiempos: Sequence[float] | None = None,
    a: float = -0.9,
    b: float = 10.0,
    tol: float = 1e-12,
) -> float:
    """Tasa interna de retorno por bisección: la r que hace VP(flujos, r) = 0.

    Supone un solo cambio de signo en los flujos (una sola raíz en [a, b]).
    """
    fa = valor_presente_flujos(flujos, a, tiempos)
    fb = valor_presente_flujos(flujos, b, tiempos)
    if fa * fb > 0:
        raise ValueError("El VP no cambia de signo en [a, b]: no hay TIR en ese intervalo.")
    while b - a > tol:
        c = (a + b) / 2
        fc = valor_presente_flujos(flujos, c, tiempos)
        if fa * fc <= 0:
            b = c
        else:
            a, fa = c, fc
    return (a + b) / 2

In [ ]:
print(f"TIR por bisección:       {tir(flujos):.6%}")
print(f"TIR con numpy-financial: {npf.irr(flujos):.6%}")

### 3.4 Ejercicio en equipos (25 minutos): ¿meses sin intereses o descuento de contado?

Fue el Buen Fin. Una computadora cuesta **24,000 pesos** y la tienda te ofrece dos opciones:

- **Contado:** 10% de descuento. Pagas 21,600 pesos hoy.
- **12 meses sin intereses:** 12 pagos mensuales de 2,000 pesos, el primero dentro de un mes.

"Sin intereses" suena a gratis. No lo es: al elegir los meses renuncias al descuento, y eso equivale a pedir prestados 21,600 pesos.

**Organización del equipo.** Quien tiene perfil financiero plantea los flujos y explica la decisión; quien tiene perfil técnico escribe el código. Todos deben poder explicar el resultado.

**Parte A.** Escribe los flujos desde el punto de vista de quien elige los meses: hoy **recibe** 21,600 (el dinero que no pagó) y después **paga** 2,000 cada mes. Calcula la TIR mensual y conviértela a tasa efectiva anual.

In [ ]:
# Tu código aquí
print(f"TIR mensual: {tir_mensual:.4%}")
print(f"Costo efectivo anual de elegir los meses: {costo_anual:.2%}")

In [ ]:
# Comprueba tu respuesta
assert math.isclose(costo_anual, 0.2183, abs_tol=1e-4)
print("Correcto.")

**Parte B.** Ahora del otro lado: si en lugar de pagar de contado inviertes tu dinero en CETES a 28 días (6.38% efectivo anual), ¿cuánto valen hoy los 12 pagos de 2,000? Convierte la tasa anual a mensual con $(1 + r_{ef})^{1/12} - 1$ y calcula el valor presente de los pagos.

In [ ]:
# Tu código aquí
print(f"Valor presente de los 12 pagos: {vp_pagos:,.2f}")
print("Precio de contado:              21,600.00")

In [ ]:
# Comprueba tu respuesta
assert 23_200 < vp_pagos < 23_250
print("Correcto.")

**Parte C.** ¿Qué le recomiendan a alguien que tiene el dinero para pagar de contado? ¿Y a alguien que no lo tiene?

*Escribe aquí tu respuesta.*

### 3.5 Reto en clase (opcional): el precio del bono a 100 años

El bono de 2110 paga un cupón de 5.750% anual en dos pagos semestrales: **2.875** dólares cada semestre por cada 100 de valor nominal, durante **200 semestres**, y devuelve los 100 al final. El mercado lo descontó al **3.05% semestral**.

Calcula su precio con `valor_presente_flujos`. El prospecto del bono reporta **94.276**.

In [ ]:
# Tu código aquí
print(f"Precio del bono: {precio_bono:.3f}")

In [ ]:
# Comprueba tu respuesta
assert math.isclose(precio_bono, 94.276, abs_tol=5e-3)
print("Correcto: valuaste un bono soberano real con una función de seis líneas.")

---
## Bloque 4 · Taller de la Práctica 2

Lo que hiciste hoy es más de la mitad de la tarea. El enunciado completo está en [`practicas/practica_02`](https://github.com/nana-bug/m1-python-finanzas-2026-2/blob/main/practicas/practica_02/README.md). En el taller avanzamos cuatro pasos:

**Paso 1 · Copia tus funciones a `src/finanzas.py` de tu equipo.** No las vuelvas a escribir: cópialas de este notebook y revisa que el nombre sea exactamente el de la tabla. Las siguientes sesiones importan estas funciones por nombre.

| Función | Fórmula | Se prueba con |
|---|---|---|
| `valor_futuro` | $VP\,(1 + r/m)^{m n}$ | 100,000 al 10%, 10 años, continua: $100{,}000\,e$ |
| `valor_presente` | $VF / (1 + r/m)^{m n}$ | 100 en 100 años al 6.1% semestral: 0.2457 |
| `tasa_efectiva` | $(1 + r/m)^{m} - 1$ | 12% capitalizable mensual: 12.68% |
| `precio_cete` | $VN / (1 + r\,d/360)$ | Banxico, 91 días al 4.39%: 9.8902485 |
| `rendimiento_cete` | $(VN/P - 1)\,360/d$ | Banxico, de 9.8902485 a 4.39% |
| `tasa_efectiva_cete` | $(1 + r\,d/360)^{360/d} - 1$ | 28 días al 6.20%: 6.38% |

**Paso 2 · Corre las seis pruebas.** Una función no está terminada hasta que pasa su prueba. La celda de abajo prueba las funciones de este notebook; en tu repositorio, la misma celda importa las funciones desde `src/finanzas.py`.

In [ ]:
assert math.isclose(valor_futuro(100_000, 0.10, 10, m=None), 100_000 * math.e)
assert math.isclose(valor_presente(100, 0.061, 100, m=2), 0.2457, abs_tol=1e-4)
assert math.isclose(tasa_efectiva(0.12, 12), 0.1268, abs_tol=1e-4)
assert round(precio_cete(0.0439, 91), 7) == 9.8902485
assert math.isclose(rendimiento_cete(9.8902485, 91), 0.0439, abs_tol=1e-7)
assert math.isclose(tasa_efectiva_cete(0.0620, 28), 0.0638, abs_tol=1e-4)
print("Las seis pruebas pasan")

**Paso 3 · La tasa libre de riesgo de tu caso.** Cada equipo documenta en su README la tasa libre de riesgo que usará todo el módulo: CETES a 28 días de la última subasta, convertida a efectiva anual y después a tasa por día hábil (252 días al año):

$$r_{\text{diaria}} = (1 + r_{ef})^{1/252} - 1$$

Esa tasa diaria es la que restarán a los rendimientos de su activo cuando calculemos el índice de Sharpe en la sesión 6.

In [ ]:
def tasa_por_periodo(r_ef: float, periodos: int = 252) -> float:
    """Tasa por periodo equivalente a una efectiva anual: (1 + r_ef)^(1/periodos) - 1."""
    return (1 + r_ef) ** (1 / periodos) - 1

In [ ]:
tasa_libre_anual = tasa_efectiva_cete(TASAS_CETES[28], 28)
tasa_libre_diaria = tasa_por_periodo(tasa_libre_anual, 252)
print(f"Subasta del {FECHA_SUBASTA} · CETES a 28 días: {TASAS_CETES[28]:.2%}")
print(f"Tasa libre de riesgo efectiva anual: {tasa_libre_anual:.2%}")
print(f"Tasa libre de riesgo por día hábil:  {tasa_libre_diaria:.4%}")

**Paso 4 · El commit lo hace otra persona.** Quien no ha subido cambios al repositorio del equipo hace el commit de hoy desde GitHub Desktop. Así el historial muestra la participación de todos.

---
## Cierre

**Lo que te llevas hoy**

- El dinero tiene fecha: 100 pesos de hoy y 100 pesos de dentro de un año no son la misma cosa.
- Para comparar tasas hay que llevarlas a la misma unidad: la tasa efectiva anual.
- Un CETE se valúa con interés simple y año de 360 días; nuestra función reproduce el ejemplo oficial de Banco de México.
- VPN y TIR son la misma fórmula vista desde dos lados.

**Para la próxima sesión.** Una acción pasa de 100 a 150 y luego baja a 75. Subió 50% y bajó 50%: ¿en cuánto quedó tu rendimiento? De esa trampa salen los rendimientos logarítmicos.

**Lecturas opcionales** (no se evalúan)

- Banco de México, [Descripción técnica de los CETES](https://www.banxico.org.mx/elib/mercado-valores-gub/OEBPS/Rsc/anexo0201.pdf). Tres páginas; de ahí salen las fórmulas y el ejemplo de hoy.
- [Prospecto del bono mexicano con vencimiento en 2110](https://capedge.com/filing/101368/0001193125-10-225080/424B2). En la primera página están el cupón, el vencimiento y el precio de 94.276%.